In [ ]:
# from google.colab import drive
# drive.flush_and_unmount()

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/CITY_FMP

/content/drive/MyDrive/CITY_FMP


In [ ]:
!ls

 config.py		    __pycache__		      wandb
 config.yaml		    ReplacementFiles	      yolo11n.pt
 controller.ipynb	    requirements.txt	      yolo12n.pt
 dataset		    runs		      yolov12
 dataset.py		   'Tomato Training Output'   YOLOv8
'disease data'		    trainDetector.py	      yolov8n.pt
'Disease Training Output'   train.py
 logger.py		    utilities.py


In [ ]:
!python --version

Python 3.12.11


In [ ]:
!nvcc --version
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
Mon Sep 22 10:35:31 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8       

In [ ]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.0 MB/s eta 0:00:00


In [ ]:
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/conv.py
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/12/yolo12.yaml
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/cfg/default.yaml
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/head.py
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/utils/loss.py

/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/conv.py
/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/12/yolo12.yaml
/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py
/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/default.yaml
/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/head.py
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/loss.py


In [ ]:
#replaces necessary file components automatically
# file_path_trainer = os.path.join(os.getcwd(), 'yolov12', 'ultralytics', 'engine', 'trainer.py' )
# file_path_default = os.path.join(os.getcwd(), 'yolov12', 'ultralytics', 'cfg', 'defaut' )

file_path_trainer = '/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py'
file_path_default = '/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/default.yaml'
file_path_conv = '/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/conv.py'
file_path_v12 = '/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/12/yolo12.yaml'
file_path_head = '/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/head.py'
file_path_loss = '/usr/local/lib/python3.12/dist-packages/ultralytics/utils/loss.py'

with open('/content/drive/MyDrive/CITY_FMP/ReplacementFiles/trainer.py', 'r') as trainer:
    replace_trainer = str(trainer.read())
with open('/content/drive/MyDrive/CITY_FMP/ReplacementFiles/default.yaml', 'r') as default:
    replace_default = str(default.read())
with open('/content/drive/MyDrive/CITY_FMP/ReplacementFiles/conv.py', 'r') as conv:
    replace_conv = str(conv.read())
with open('/content/drive/MyDrive/CITY_FMP/ReplacementFiles/yolov12.yaml', 'r') as v12:
    replace_v12 = str(v12.read())
with open('/content/drive/MyDrive/CITY_FMP/ReplacementFiles/loss.py', 'r') as loss:
    replace_loss = str(loss.read())

new_head = """
class ReDesignedDetectionHead(nn.Module):
    stride = None
    dynamic = False
    export = False

    def __init__(self, nc=80, ch=(), groups=4):
        super().__init__()
        self.nc = nc
        self.no = nc + 5
        self.nl = len(ch)

        self.shared1 = nn.Conv2d(ch[0], ch[0], 3, 1, 1, groups=groups, bias=False)
        self.shared2 = nn.Conv2d(ch[0], ch[0], 3, 1, 1, groups=groups, bias=False)

        self.proj = nn.ModuleList([nn.Conv2d(c, ch[0], 1, 1, 0) for c in ch])
        self.m = nn.ModuleList([nn.Conv2d(ch[0], self.no, 1, 1, 0) for _ in ch])

    def forward(self, x):
        z = []
        for i, f in enumerate(x):
            f = self.proj[i](f)
            f = self.shared1(f)
            f = self.shared2(f)
            z.append(self.m[i](f))
        return z
"""

with open(file_path_trainer, 'w') as fT:
    fT.write(replace_trainer)
    print("Trainer successfully replaced")
with open(file_path_default, 'w') as fD:
    fD.write("---\n" + replace_default)
    print("Default successfully replaced")
# with open(file_path_conv, 'w') as fC:
#     fC.write(replace_conv)
#     print("Conv successfully replaced")
# with open(file_path_v12, 'w') as fV:
#     fV.write(replace_v12)

#     print("YOLOv12 successfully replaced")
# with open(file_path_head, 'a') as fH:
#     fH.write("\n" + new_head)

#     print("Head successfully replaced")
# with open(file_path_loss, 'w') as fL:
#     fL.write(replace_loss)
#     print("Loss successfully replaced")

Trainer successfully replaced
Default successfully replaced


In [ ]:
!yolo settings wandb=True

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Updated 'wandb=True'
JSONDict("/root/.config/Ultralytics/settings.json"):
{
  "settings_version": "0.0.6",
  "datasets_dir": "/content/drive/MyDrive/CITY_FMP/datasets",
  "weights_dir": "weights",
  "runs_dir": "runs",
  "uuid": "569f3ba64b326db489132663f79cd37279811de477381b83ac131e6cdd129cbb",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "hub": true,
  "mlflow": true,
  "neptune": true,
  "raytune": true,
  "tensorboard": false,
  "wandb": true,
  "vscode_msg": true,
  "openvino_msg": true
}
💡 Learn more about Ultralytics Settings at https://docs.ultralytics.com/quickstart/#ultralytics-settings


In [ ]:
#4dbce09ff68bb778f03c435cfda8289ee37a28e2
!python -u train.py

Selected Device: cuda
Tesla T4
PyTorch version: 2.8.0+cu126
CUDA available: True
CUDA version: 12.6
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: 2
wandb: You chose 'Use an existing W&B account'
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: unliveddisc03 (unliveddisc03-city-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.21.3
wandb: Run data is saved locally in /content/d